<a href="https://colab.research.google.com/github/imabari/ImabariScraping/blob/master/%E3%82%AF%E3%83%AA%E3%83%BC%E3%83%8B%E3%83%B3%E3%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ

愛媛県オープンデータカタログのクリーニング所一覧からWEBAPIを使って一覧表を作成

https://www.pref.ehime.jp/opendata-catalog/dataset/2310.html

# WEBAPI

データの一覧を取得

In [1]:
import requests

In [2]:
url = "https://www.pref.ehime.jp/opendata-catalog/api/package_show?id=b2c0dd49-8dc7-433d-9271-bd016932fd2f"

In [3]:
r = requests.get(url)
r.raise_for_status()

# JSON

一覧表にして確認

In [4]:
import pandas as pd

In [5]:
pd.options.plotting.backend = "plotly"

In [6]:
data = r.json()["result"]["resources"]

In [7]:
df0 = pd.json_normalize(data).sort_index(ascending=False)
df0 = df0[df0["format"] == "XLS"]
df0

,id,uuid,revision_id,state,name,filename,text,rdf_iri,rdf_error,created,updated,download_url,url,format,license.id,license.name,license.uid,license.state,license.created,license.updated
3,13642,a32c1cae-953a-4e55-8113-fccb7715cab4,ca2eedaf-078f-4d85-b1a8-539c506d0c11,public,クリーニング所施設一覧（令和５年12月末時点）.xls,クリーニング所施設一覧（令和５年12月末時点）.xls,None,None,None,2024-01-16T11:44:14.343+09:00,2024-01-16T11:44:14.348+09:00,https://www.pref.ehime.jp/opendata-catalog/dat...,https://www.pref.ehime.jp//opendata-catalog/fs...,XLS,1,表示（CC BY）,None,public,2019-01-09T18:07:06.926+09:00,2024-03-19T16:24:03.538+09:00
2,13721,a3e03a25-d28d-42df-8094-3f40a23aa372,20932af5-803d-4d14-955b-632b8eb5292a,public,クリーニング所施設一覧（令和６年１月新規）.xls,クリーニング所施設一覧（令和６年１月新規）.xls,None,None,None,2024-02-19T14:55:51.798+09:00,2024-02-19T14:55:51.811+09:00,https://www.pref.ehime.jp/opendata-catalog/dat...,https://www.pref.ehime.jp//opendata-catalog/fs...,XLS,1,表示（CC BY）,None,public,2019-01-09T18:07:06.926+09:00,2024-03-19T16:24:03.538+09:00
1,13774,f67688a3-5006-4c75-9dd6-a8f995f69752,116bc430-1a06-46ff-9417-93c977b54a10,public,クリーニング所施設一覧（令和６年２月新規）.xls,クリーニング所施設一覧（令和６年２月新規）.xls,None,None,None,2024-03-15T17:14:18.562+09:00,2024-04-17T17:14:08.558+09:00,https://www.pref.ehime.jp/opendata-catalog/dat...,https://www.pref.ehime.jp//opendata-catalog/fs...,XLS,1,表示（CC BY）,None,public,2019-01-09T18:07:06.926+09:00,2024-03-19T16:24:03.538+09:00
0,14086,142bb161-bd7b-42c1-95dc-0eed439e52eb,fdca9634-9130-49fb-8631-3c4b97b5201a,public,クリーニング所施設一覧（令和６年３月新規）.xls,クリーニング所施設一覧（令和６年３月新規）.xls,None,None,None,2024-04-17T17:11:05.564+09:00,2024-04-17T17:11:05.566+09:00,https://www.pref.ehime.jp/opendata-catalog/dat...,https://www.pref.ehime.jp//opendata-catalog/fs...,XLS,1,表示（CC BY）,None,public,2019-01-09T18:07:06.926+09:00,2024-03-19T16:24:03.538+09:00


# データをダウンロード

In [8]:
# ダウンロード先URLをリストに変換
links = df0["download_url"].tolist()

In [9]:
dfs = []

for link in links:

    df = pd.read_excel(link, header=None)

    # ヘッダーの位置が違うので判定
    skiprows = df.iat[1, 0].startswith("施設名称")

    df = pd.read_excel(link, skiprows=int(skiprows))

    dfs.append(df)

In [10]:
# 各ファイルを結合（愛媛県内全部）
df1 = pd.concat(dfs, ignore_index=True)
df1

,施設名称,施設郵便番号,施設所在地１,施設所在地２,施設電話番号,営業者氏名,営業者役職名,営業者その他役職名,代表者氏名,検査確認番号記号,検査確認番号,許可交付日,クリーニング種別
0,ワタキューセイモア株式会社 松山工場,791-3162,伊予郡松前町出作５２８－１,NaN,089-984-0171,ワタキューセイモア株式会社,代表取締役,NaN,村田 清和,松保生18,第2号,H19.02.23,一般クリーニング所
1,くりーにんぐ えびす屋,791-3132,伊予郡松前町西高柳２２０－３,NaN,089-984-3063,髙田 洋一,NaN,NaN,NaN,松保生15,第3号,H16.01.08,一般クリーニング所
2,石田クリーニングキラ,791-3110,伊予郡松前町大字浜１０８９－１,NaN,089-984-6570,石田クリーニング（株）,代表取締役,NaN,清本 有策,松保生16,第4号,H17.01.26,取次所
3,有限会社創美舎カーテン工場,791-3120,伊予郡松前町筒井1320-1,NaN,089-997-3620,有限会社創美舎,代表取締役,NaN,綿村 俊宏,4中局生,第2203001号,R04.09.30,一般クリーニング所
4,清水屋ばんどう店,791-3120,伊予郡松前町筒井１５７５－４,NaN,089-984-1116,阪東 秀人,NaN,NaN,NaN,松保生15,第4号,H16.03.26,取次所
...,...,...,...,...,...,...,...,...,...,...,...,...,...
587,フジクリーニング目黒店,798-2106,北宇和郡松野町目黒３８８,NaN,43-0288,二宮 嗣,NaN,NaN,NaN,宇保衛,第282号,S62.09.02,取次所
588,西岡クリーニング エーマックス店,798-4110,南宇和郡愛南町御荘平城791-1,NaN,,有限会社西岡クリーニング店,代表取締役,NaN,西岡 信夫,5南局生,第2303001号,R06.01.29,取次所
589,ヤングドライ フジ店,792-0802,新居浜市新須賀町2丁目10－7,NaN,NaN,株式会社ヤングドライ新居浜,代表取締役,NaN,永田 真一,5東生,第2303004号,R06.01.31,取次所
590,ヤングドライ 神郷店,792-0888,新居浜市田の上１丁目5－50,NaN,NaN,株式会社ヤングドライ新居浜,NaN,NaN,NaN,5東生,第2303005号,R06.02.15,取次所


In [11]:
# 今治市のみ抽出
df2 = df1[df1["施設所在地１"].str.startswith("今治市")].copy().reset_index(drop=True)
df2

,施設名称,施設郵便番号,施設所在地１,施設所在地２,施設電話番号,営業者氏名,営業者役職名,営業者その他役職名,代表者氏名,検査確認番号記号,検査確認番号,許可交付日,クリーニング種別
0,清水屋イオンモール今治新都市店,794-0068,今治市にぎわい広場1番地1,NaN,0898-35-3613,株式会社清水屋,代表取締役,NaN,清水 栄治,5東今生,第2303001号,R05.10.27,取次所
1,清水屋クリーニング 阿方店,794-0081,今治市阿方字山之間甲３８１－３,NaN,0898-32-2037,有限会社 アタル,代表取締役,NaN,清水 栄治,今保18,第1029号,H18.12.01,取次所
2,ワシ屋クリーニングクリーニングドクター衣干店,794-0813,今治市衣干町４丁目６１－１,NaN,32-3053,株式会社ワシ屋グループ,代表取締役,NaN,村上 康,今保18,第1043号,H19.03.26,取次所
3,IKEUCHI ORGANIC TOWEL CLINIC,794-0084,今治市延喜甲762番地,NaN,0898-31-2255,IKEUCHI ORGANIC株式会社,代表取締役,NaN,池内 計司,29東今生,第1703001号,H29.11.07,一般クリーニング所
4,つかさクリーニング相原ストアー,794-0814,今治市横田町１丁目５－５,NaN,,渡辺 司,NaN,NaN,NaN,今保,第167号,S57.03.12,取次所
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,朝日屋クリーニング橋田取次所,794-0063,今治市片山６３,NaN,,長野 のぶ子,NaN,NaN,NaN,今保,第152号,S43.12.16,取次所
81,カガワ屋クリーニング鳥生取次所,794-0812,今治市北高下町２丁目１－５０,NaN,0898-32-6333,森川 義夫,NaN,NaN,NaN,今保,第184号,S54.03.05,取次所
82,大黒屋,794-0054,今治市北日吉町１丁目１１－１６,NaN,0898-22-5585,馬越 直人,NaN,NaN,NaN,今保,第72号,S41.05.02,一般クリーニング所
83,JRクレメントイン今治,794-0028,今治市北宝来町2丁目甲773番9,NaN,0898-55-8333,株式会社JR四国ホテルズ,代表取締役,NaN,矢田 栄一,3東今生,第2103002号,R03.08.10,取次所


In [12]:
# 重複確認
df2[df2.duplicated()]

,施設名称,施設郵便番号,施設所在地１,施設所在地２,施設電話番号,営業者氏名,営業者役職名,営業者その他役職名,代表者氏名,検査確認番号記号,検査確認番号,許可交付日,クリーニング種別


# 可視化

In [13]:
df2["クリーニング種別"].value_counts().plot.barh()

In [14]:
df2["営業者氏名"].str.replace("\s", "", regex=True).value_counts().head(10).plot.barh()

In [15]:
df2.to_csv("クリーニング.csv", encoding="utf_8_sig")

In [16]:
from google.colab import files

files.download("クリーニング.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>